
# Generative Modeling by Estimating Gradients of the Data Distribution：完整理论学习笔记

**论文：** Generative Modeling by Estimating Gradients of the Data Distribution  
**作者：** Yang Song、Stefano Ermon  
**会议：** NeurIPS 2019  
**论文地址：** https://arxiv.org/abs/1907.05600  
**核心方法：** Noise Conditional Score Network（NCSN）与 Annealed Langevin Dynamics（ALD）

---

# 第一部分：概率论基础

在理解 NCSN 之前，需要掌握几个基本概念：

1. 随机变量的概率分布与概率密度。
2. 独立随机变量的线性组合。
3. 随机变量求和与概率密度卷积。
4. 条件分布、边缘分布和联合分布。
5. 贝叶斯后验与条件期望。

这些概念是后面高斯加噪、DSM 和 Score 推导的数学基础。

## 一、随机变量与概率密度

### 1.1 随机变量和概率密度的区别

假设随机变量：

$$
X\sim p_X(x)
$$

其中：

- $X$：随机变量。
- $x$：随机变量的一次具体取值。
- $p_X(x)$：随机变量 $X$ 的概率密度函数。

例如：

$$
X\sim\mathcal N(0,1)
$$

表示 $X$ 服从标准高斯分布。

一次采样可能得到：

$$
x=0.5
$$

另一次可能得到：

$$
x=-1.2
$$

概率密度满足：

$$
p_X(x)\geq0
$$

以及归一化条件：

$$
\boxed{
\int_{\mathbb R^D}p_X(x)\,dx=1
}
$$

对于连续随机变量，$p_X(x)$ 是概率密度，而不是某个点的概率。

某一区域 $A$ 内的概率为：

$$
\boxed{
P(X\in A)=\int_Ap_X(x)\,dx
}
$$

### 1.2 随机变量经过函数变换

假设：

$$
Y=f(X)
$$

那么 $Y$ 的分布由 $X$ 的分布和变换 $f$ 共同决定。

例如：

$$
Y=aX+b
$$

当 $a\neq0$，一维概率密度满足：

$$
\boxed{
p_Y(y)=\frac1{|a|}p_X\left(\frac{y-b}{a}\right)
}
$$

其中：

$$
\frac1{|a|}
$$

来自变量替换时的 Jacobian。

如果：

$$
X\sim\mathcal N(\mu,\sigma^2)
$$

那么：

$$
\boxed{
aX+b\sim\mathcal N(a\mu+b,a^2\sigma^2)
}
$$

这意味着，对高斯随机变量进行线性变换后，结果仍然是高斯随机变量。

---

## 二、独立随机变量的线性组合

这是理解扩散模型中高斯加噪最重要的概率论基础。

### 2.1 两个独立高斯随机变量相加

假设：

$$
X\sim\mathcal N(\mu_X,\sigma_X^2)
$$

$$
Y\sim\mathcal N(\mu_Y,\sigma_Y^2)
$$

且：

$$
X\perp Y
$$

定义：

$$
Z=X+Y
$$

那么：

$$
\boxed{
Z\sim\mathcal N
\left(
\mu_X+\mu_Y,\,
\sigma_X^2+\sigma_Y^2
\right)
}
$$

也就是说：

**独立高斯随机变量相加，均值相加，方差相加。**

进一步，对于：

$$
Z=aX+bY+c
$$

有：

$$
\boxed{
Z\sim\mathcal N
\left(
a\mu_X+b\mu_Y+c,\,
a^2\sigma_X^2+b^2\sigma_Y^2
\right)
}
$$

这里方差能够直接相加，依赖于独立性。

如果 $X,Y$ 不独立，那么：

$$
\operatorname{Var}(aX+bY)
=
a^2\operatorname{Var}(X)
+b^2\operatorname{Var}(Y)
+2ab\operatorname{Cov}(X,Y)
$$

### 2.2 多维高斯随机变量的线性组合

假设：

$$
X\sim\mathcal N(\mu_X,\Sigma_X)
$$

$$
Y\sim\mathcal N(\mu_Y,\Sigma_Y)
$$

且两者独立。

定义：

$$
Z=AX+BY+c
$$

那么：

$$
\boxed{
Z\sim\mathcal N
\left(
A\mu_X+B\mu_Y+c,\,
A\Sigma_XA^\top+B\Sigma_YB^\top
\right)
}
$$

这也是扩散模型中经常出现：

$$
X_t=\alpha_tX_0+\sigma_t\epsilon
$$

的原因。

不过需要注意：

**如果 $X_0$ 本身不是高斯随机变量，那么 $X_t$ 的边缘分布一般也不是高斯分布。**

只有给定 $X_0=x_0$ 时：

$$
\boxed{
X_t\mid X_0=x_0
\sim
\mathcal N(\alpha_tx_0,\sigma_t^2I)
}
$$

这一点对 DSM 非常重要。

---

## 三、随机变量求和与概率密度卷积

### 3.1 什么是卷积？

对于两个函数：

$$
f(x),\quad g(x)
$$

卷积定义为：

$$
\boxed{
(f*g)(z)
=
\int_{\mathbb R^D}f(x)g(z-x)\,dx
}
$$

卷积的直观意义是：

**计算两个函数在不同相对位置下的加权重叠程度。**

在概率论中，卷积有一个更加明确的含义：

> 两个独立随机变量相加，其概率密度等于各自概率密度的卷积。

### 3.2 为什么随机变量相加对应卷积？

假设：

$$
X\sim p_X(x)
$$

$$
Y\sim p_Y(y)
$$

且：

$$
X\perp Y
$$

定义：

$$
Z=X+Y
$$

我们希望求：

$$
p_Z(z)
$$

由于：

$$
Z=X+Y
$$

当：

$$
Z=z
$$

时，所有满足：

$$
x+y=z
$$

的组合，都可能产生结果 $z$。

也就是：

$$
y=z-x
$$

因为 $X,Y$ 独立，所以联合密度为：

$$
p_{X,Y}(x,y)=p_X(x)p_Y(y)
$$

对变量进行变换：

$$
(x,y)\rightarrow(x,z=x+y)
$$

其 Jacobian 行列式的绝对值为 1。

因此：

$$
p_{X,Z}(x,z)=p_X(x)p_Y(z-x)
$$

为了得到 $Z$ 的边缘分布，需要对 $x$ 积分：

$$
\boxed{
p_Z(z)
=
\int_{\mathbb R^D}
p_X(x)p_Y(z-x)\,dx
}
$$

这正是卷积：

$$
\boxed{
p_Z=p_X*p_Y
}
$$

注意，这里有一个必要条件：

$$
\boxed{X\perp Y}
$$

如果两个随机变量不独立，就不能简单地使用两个边缘密度的卷积。

### 3.3 高斯加噪为什么是卷积？

现在考虑 NCSN 中的加噪过程：

$$
\boxed{
X=X_0+\sigma\epsilon
}
$$

其中：

$$
X_0\sim p_{\mathrm{data}}
$$

$$
\epsilon\sim\mathcal N(0,I)
$$

并且：

$$
X_0\perp\epsilon
$$

定义噪声变量：

$$
N=\sigma\epsilon
$$

则：

$$
N\sim\mathcal N(0,\sigma^2I)
$$

所以：

$$
X=X_0+N
$$

根据随机变量求和的卷积公式：

$$
\boxed{
p_X(x)
=
p_{\mathrm{data}}*
\mathcal N(0,\sigma^2I)
}
$$

也就是：

$$
\boxed{
p_\sigma(x)
=
\int
p_{\mathrm{data}}(x_0)
\mathcal N(x;x_0,\sigma^2I)\,dx_0
}
$$

这里：

- $p_{\mathrm{data}}$：干净数据分布。
- $\mathcal N(0,\sigma^2I)$：噪声分布。
- $p_\sigma$：加噪后的边缘分布。

因此：

**高斯加噪不仅仅是在样本数值上添加随机扰动，从概率分布角度看，它相当于对原始数据分布进行高斯卷积。**

### 3.4 为什么高斯卷积具有平滑作用？

考虑最简单的情况。

假设原始数据只有一个固定位置：

$$
X_0=3
$$

那么它的概率分布可以表示为：

$$
p_{\mathrm{data}}(x)=\delta(x-3)
$$

其中 $\delta$ 是 Dirac 测度的形式化密度表示。

添加高斯噪声：

$$
X=3+\sigma\epsilon
$$

得到：

$$
\boxed{
X\sim\mathcal N(3,\sigma^2)
}
$$

原来所有概率质量都集中在 $x=3$。

加噪之后，概率质量扩散到 $3$ 附近。

当 $\sigma$ 增大：

- 高斯峰值降低。
- 分布宽度增大。
- 概率质量分布到更远的位置。

如果原始分布由多个尖峰组成：

$$
p_{\mathrm{data}}(x)
=
\sum_{i=1}^K\pi_i\delta(x-\mu_i)
$$

那么加噪后：

$$
\boxed{
p_\sigma(x)
=
\sum_{i=1}^K
\pi_i\mathcal N(x;\mu_i,\sigma^2I)
}
$$

原始尖峰就变成了多个高斯峰。

随着噪声增大，这些高斯峰可能逐渐重叠，甚至形成一个更加平滑的单峰分布。

这正是 NCSN 使用多尺度高斯噪声的数学基础。

---

## 四、条件分布、边缘分布与贝叶斯后验

这一部分直接决定我们能否正确理解 DSM。

### 4.1 联合分布

假设：

$$
X_0\sim p_{\mathrm{data}}
$$

并且：

$$
X\mid X_0=x_0
\sim\mathcal N(x_0,\sigma^2I)
$$

那么联合分布为：

$$
\boxed{
p(x_0,x)
=
p_{\mathrm{data}}(x_0)p(x\mid x_0)
}
$$

它描述干净样本和加噪样本共同出现的概率关系。

### 4.2 边缘分布

对干净数据变量积分：

$$
\boxed{
p_\sigma(x)
=
\int p(x_0,x)\,dx_0
}
$$

也就是：

$$
\boxed{
p_\sigma(x)
=
\int
p_{\mathrm{data}}(x_0)
p(x\mid x_0)\,dx_0
}
$$

这里得到的是加噪样本 $X$ 的边缘分布。

### 4.3 后验分布

反过来，如果我们已经观察到加噪样本：

$$
X=x
$$

那么它可能来自哪些干净样本？

这个问题由后验分布回答：

$$
\boxed{
p(x_0\mid x)
=
\frac{
p(x\mid x_0)p_{\mathrm{data}}(x_0)
}{
p_\sigma(x)
}
}
$$

这里：

- $p_{\mathrm{data}}(x_0)$：干净数据先验。
- $p(x\mid x_0)$：高斯加噪似然。
- $p(x_0\mid x)$：给定加噪样本后的干净数据后验。

它告诉我们：

**一个加噪样本可能对应多个不同的干净样本，并且这些干净样本的可能性通常不同。**

### 4.4 条件期望

对于函数：

$$
f(X_0,X)
$$

给定：

$$
X=x
$$

其条件期望为：

$$
\boxed{
\mathbb E[f(X_0,X)\mid X=x]
=
\int
f(x_0,x)p(x_0\mid x)\,dx_0
}
$$

这个期望不是对所有干净数据进行均匀平均，而是根据后验概率加权平均。

这就是后面 DSM 为什么能够学习边缘 Score 的关键。

---

# 第二部分：随机微分方程 SDE 的数学基础

## 五、从 ODE 到 SDE

### 5.1 ODE 描述确定性动力学

普通微分方程：

$$
\boxed{
\frac{dx_t}{dt}=f(x_t,t)
}
$$

表示状态 $x_t$ 随时间变化的规律。

如果给定初始条件：

$$
x_0
$$

那么在适当条件下，可以确定唯一的演化轨迹。

例如：

$$
\frac{dx_t}{dt}=-x_t
$$

解析解为：

$$
\boxed{
x_t=e^{-t}x_0
}
$$

因此：

$$
x_t\rightarrow0
$$

所有初始位置都会逐渐收敛到零点。

### 5.2 SDE 引入随机扰动

现在考虑：

$$
\boxed{
dX_t=f(X_t,t)\,dt+g(t)\,dW_t
}
$$

其中：

- $X_t$：随机过程。
- $f$：漂移系数。
- $g$：扩散系数。
- $W_t$：标准布朗运动。

它与 ODE 最大的区别是：

**即使初始状态相同，不同的布朗运动轨迹也会产生不同的采样结果。**

因此，SDE 描述的不是唯一确定轨迹，而是一族随机轨迹。

---

## 六、布朗运动及其性质

标准布朗运动满足：

$$
W_0=0
$$

对于：

$$
0\leq s<t
$$

有：

$$
\boxed{
W_t-W_s\sim\mathcal N(0,(t-s)I)
}
$$

并且不同、不重叠时间区间上的增量相互独立。

这意味着：

$$
\mathbb E[W_t-W_s]=0
$$

$$
\operatorname{Cov}(W_t-W_s)=(t-s)I
$$

如果时间间隔为：

$$
\Delta t
$$

那么：

$$
\boxed{
\Delta W=\sqrt{\Delta t}\,\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

因此，SDE 中的随机项：

$$
g(t)dW_t
$$

在数值离散化时，对应：

$$
\boxed{
g(t)\sqrt{\Delta t}\,\epsilon
}
$$

这里需要特别注意：

**布朗运动的增量标准差与时间步长的平方根成正比，而不是与时间步长本身成正比。**

---

## 七、Euler–Maruyama：如何数值求解 SDE？

考虑：

$$
dX_t=f(X_t,t)\,dt+g(t)dW_t
$$

在时间区间：

$$
[t_k,t_{k+1}]
$$

上积分：

$$
X_{t_{k+1}}-X_{t_k}
=
\int_{t_k}^{t_{k+1}}f(X_t,t)\,dt
+
\int_{t_k}^{t_{k+1}}g(t)\,dW_t
$$

设：

$$
h=t_{k+1}-t_k
$$

使用 Euler–Maruyama 近似：

$$
\int_{t_k}^{t_{k+1}}f(X_t,t)\,dt
\approx
hf(X_{t_k},t_k)
$$

如果扩散系数在一步内近似为常数：

$$
\int_{t_k}^{t_{k+1}}g(t)\,dW_t
\approx
g(t_k)\sqrt h\,\epsilon_k
$$

因此：

$$
\boxed{
X_{k+1}
=
X_k+
hf(X_k,t_k)
+
g(t_k)\sqrt h\,\epsilon_k
}
$$

其中：

$$
\epsilon_k\sim\mathcal N(0,I)
$$

这就是 SDE 最常见的数值积分方法之一。

需要强调：

**数值求解 SDE，得到的是随机过程的一条采样轨迹，而不是直接得到概率密度的解析表达式。**

如果独立运行大量随机轨迹，那么在同一时间 $t$ 的样本集合，就可以近似随机变量 $X_t$ 的概率分布。

---

## 八、Fokker–Planck 方程：从样本运动到概率分布运动

### 8.1 SDE 与概率密度的关系

考虑：

$$
\boxed{
dX_t=f(X_t)\,dt+g\,dW_t
}
$$

其中 $g$ 是常数。

假设随机变量 $X_t$ 在时间 $t$ 的概率密度为：

$$
q_t(x)
$$

那么：

$$
\boxed{
\frac{\partial q_t}{\partial t}
=
-\nabla\cdot(fq_t)
+
\frac{g^2}{2}\Delta q_t
}
$$

这就是 Fokker–Planck 方程。

其中：

$$
\nabla\cdot(fq_t)
$$

描述漂移引起的概率质量移动。

而：

$$
\Delta q_t
$$

描述随机扩散引起的概率密度变化。

因此：

- SDE 描述单个随机样本的运动。
- Fokker–Planck 方程描述整体概率分布的演化。

### 8.2 为什么概率密度变化的积分为零？

因为任意时间的概率密度都必须满足：

$$
\boxed{
\int q_t(x)\,dx=1
}
$$

对时间求导：

$$
\frac{d}{dt}
\int q_t(x)\,dx
=
\frac{d}{dt}1
$$

在允许交换积分与求导的条件下：

$$
\boxed{
\int
\frac{\partial q_t(x)}{\partial t}\,dx=0
}
$$

这表示：

**概率密度可以在不同位置增加或减少，但整个空间中的总概率始终保持为 1。**

它并不意味着：

$$
\frac{\partial q_t(x)}{\partial t}=0
$$

只有当概率密度在每个位置都不再随时间变化时，才称为平稳分布。

---

## 九、平稳分布与 Langevin SDE 的构造

现在假设我们希望从某个目标分布：

$$
p(x)
$$

中采样。

我们能否构造一个 SDE，使它长期运行后的分布就是 $p(x)$？

考虑：

$$
dX_t=f(X_t)\,dt+g\,dW_t
$$

对应 Fokker–Planck 方程：

$$
\frac{\partial q_t}{\partial t}
=
-\nabla\cdot(fq_t)
+
\frac{g^2}{2}\Delta q_t
$$

希望：

$$
q_t(x)=p(x)
$$

时满足：

$$
\frac{\partial q_t}{\partial t}=0
$$

代入：

$$
0=
-\nabla\cdot(fp)
+
\frac{g^2}{2}\Delta p
$$

整理：

$$
\boxed{
\nabla\cdot
\left(
fp-\frac{g^2}{2}\nabla p
\right)=0
}
$$

我们可以选择零概率流条件：

$$
fp-\frac{g^2}{2}\nabla p=0
$$

得到：

$$
f(x)=\frac{g^2}{2}\frac{\nabla p(x)}{p(x)}
$$

因为：

$$
\nabla\log p(x)=\frac{\nabla p(x)}{p(x)}
$$

所以：

$$
\boxed{
f(x)=\frac{g^2}{2}\nabla\log p(x)
}
$$

于是得到：

$$
\boxed{
dX_t=
\frac{g^2}{2}\nabla\log p(X_t)\,dt
+
g\,dW_t
}
$$

当：

$$
g=1
$$

时：

$$
\boxed{
dX_t=
\frac12\nabla\log p(X_t)\,dt+dW_t
}
$$

这就是过阻尼 Langevin SDE 的常见形式。

这里要注意两点：

第一，**Langevin SDE 是为了从目标分布中采样而构造的随机过程，不是由 NCSN 的高斯加噪直接推导出来的。**

第二，我们证明的是 $p(x)$ 是一个平稳分布。要进一步证明从任意初始分布收敛到 $p(x)$，还需要适当的遍历性与正则性条件。

---

## 十、为什么 Langevin Dynamics 能够逐渐接近目标分布？

定义：

$$
q_t(x)
$$

为 Langevin 过程当前的概率密度。

目标分布为：

$$
p(x)
$$

Langevin SDE：

$$
dX_t=
\frac12\nabla\log p(X_t)\,dt+dW_t
$$

对应：

$$
\frac{\partial q_t}{\partial t}
=
-\frac12\nabla\cdot(q_t\nabla\log p)
+
\frac12\Delta q_t
$$

因为：

$$
\Delta q_t
=
\nabla\cdot(q_t\nabla\log q_t)
$$

所以：

$$
\boxed{
\frac{\partial q_t}{\partial t}
=
\frac12\nabla\cdot
\left[
q_t\nabla\log\frac{q_t}{p}
\right]
}
$$

定义 KL 散度：

$$
D_{\mathrm{KL}}(q_t\|p)
=
\int q_t\log\frac{q_t}{p}\,dx
$$

对时间求导：

$$
\frac{d}{dt}D_{\mathrm{KL}}(q_t\|p)
=
\int
\frac{\partial q_t}{\partial t}
\left(
\log\frac{q_t}{p}+1
\right)dx
$$

利用：

$$
\int\frac{\partial q_t}{\partial t}\,dx=0
$$

得到：

$$
\frac{d}{dt}D_{\mathrm{KL}}(q_t\|p)
=
\int
\frac{\partial q_t}{\partial t}
\log\frac{q_t}{p}\,dx
$$

代入 Fokker–Planck 方程：

$$
=
\frac12
\int
\nabla\cdot
\left[
q_t\nabla\log\frac{q_t}{p}
\right]
\log\frac{q_t}{p}\,dx
$$

在边界项消失的条件下进行分部积分：

$$
\boxed{
\frac{d}{dt}D_{\mathrm{KL}}(q_t\|p)
=
-\frac12
\int
q_t(x)
\left\|
\nabla\log\frac{q_t(x)}{p(x)}
\right\|^2dx
}
$$

因此：

$$
\boxed{
\frac{d}{dt}D_{\mathrm{KL}}(q_t\|p)\leq0
}
$$

这说明，在理想连续时间动力学中，当前分布与目标分布之间的 KL 散度不会增加。

在进一步满足适当收敛条件时：

$$
\boxed{
q_t\xrightarrow[t\rightarrow\infty]{}p
}
$$

因此，Langevin Dynamics 能够通过随机采样逐渐逼近目标分布。

这里最重要的理解是：

**Score 负责推动样本向高概率密度区域移动，随机扰动负责维持概率分布的扩散，两者共同决定最终的平稳分布。**

---

# 第三部分：NCSN 论文核心理论详解

## 十一、论文的研究动机

NCSN 的基本问题是：

假设我们拥有真实数据：

$$
x_1,x_2,\ldots,x_N\sim p_{\mathrm{data}}
$$

但不知道：

$$
p_{\mathrm{data}}(x)
$$

的解析表达式。

我们能否不直接学习概率密度，而是学习：

$$
\boxed{
s(x)=\nabla_x\log p_{\mathrm{data}}(x)
}
$$

然后使用 Langevin Dynamics 进行采样？

从前面的推导可知：

$$
dX_t=
\frac12s(X_t)\,dt+dW_t
$$

在适当条件下，可以构造以目标分布为平稳分布的随机过程。

因此，问题转化为：

**如何从有限训练数据中准确学习 Score？**

这就是 Score Matching 的研究问题。

---

## 十二、Score Matching：如何在不知道概率密度的情况下学习 Score？

定义：

$$
s^*(x)=\nabla_x\log p_{\mathrm{data}}(x)
$$

我们希望最小化：

$$
\mathcal J(\theta)
=
\frac12
\mathbb E_{p_{\mathrm{data}}}
\|s_\theta(x)-s^*(x)\|^2
$$

但真实 Score 未知。

展开：

$$
\mathcal J(\theta)
=
\frac12\mathbb E_p\|s_\theta\|^2
-
\mathbb E_p[s_\theta^\top s^*]
+C
$$

其中 $C$ 不依赖模型参数。

由于：

$$
s^*=\frac{\nabla p}{p}
$$

所以：

$$
\mathbb E_p[s_\theta^\top s^*]
=
\int s_\theta^\top\nabla p\,dx
$$

对各个维度进行分部积分，并假设边界项消失：

$$
\int
s_{\theta,j}
\frac{\partial p}{\partial x_j}dx
=
-\int
p\frac{\partial s_{\theta,j}}{\partial x_j}dx
$$

因此：

$$
\boxed{
\mathbb E_p[s_\theta^\top s^*]
=
-\mathbb E_p[\nabla\cdot s_\theta]
}
$$

最终得到：

$$
\boxed{
\mathcal J_{\mathrm{SM}}(\theta)
=
\mathbb E_p
\left[
\frac12\|s_\theta(x)\|^2
+
\nabla\cdot s_\theta(x)
\right]
}
$$

这个损失不需要真实 Score。

但是：

$$
\nabla\cdot s_\theta
=
\operatorname{tr}(\nabla_xs_\theta)
$$

涉及神经网络对输入的 Jacobian，计算开销较大。

因此，论文讨论了 Sliced Score Matching 和 Denoising Score Matching。

### Sliced Score Matching

设：

$$
J_\theta(x)=\nabla_xs_\theta(x)
$$

随机采样：

$$
v\sim\mathcal N(0,I)
$$

由于：

$$
\mathbb E[vv^\top]=I
$$

因此：

$$
\boxed{
\mathbb E_v[v^\top J_\theta v]
=
\operatorname{tr}(J_\theta)
}
$$

所以可以使用随机向量估计 Jacobian 的迹。

对应损失：

$$
\boxed{
\mathcal J_{\mathrm{SSM}}
=
\mathbb E_{x,v}
\left[
v^\top J_\theta(x)v
+
\frac12\|s_\theta(x)\|^2
\right]
}
$$

这避免了显式构造完整 Jacobian。

不过 NCSN 最终主要使用 DSM 进行训练。

---

## 十三、DSM：为什么条件 Score 的监督能够学到边缘 Score？

这是整篇论文最重要的数学推导之一。

### 13.1 构造加噪分布

定义：

$$
X_0\sim p_{\mathrm{data}}
$$

$$
\epsilon\sim\mathcal N(0,I)
$$

$$
\boxed{
X=X_0+\sigma\epsilon
}
$$

因此：

$$
\boxed{
p(x\mid x_0)
=
\mathcal N(x;x_0,\sigma^2I)
}
$$

边缘分布为：

$$
\boxed{
p_\sigma(x)
=
\int
p_{\mathrm{data}}(x_0)p(x\mid x_0)\,dx_0
}
$$

我们真正希望学习：

$$
\boxed{
s^*(x,\sigma)=\nabla_x\log p_\sigma(x)
}
$$

但 $p_\sigma(x)$ 通常无法直接计算。

### 13.2 条件 Score 可以解析计算

由于：

$$
p(x\mid x_0)=\mathcal N(x;x_0,\sigma^2I)
$$

所以：

$$
\boxed{
\nabla_x\log p(x\mid x_0)
=
-\frac{x-x_0}{\sigma^2}
}
$$

又因为：

$$
x=x_0+\sigma\epsilon
$$

所以：

$$
\boxed{
\nabla_x\log p(x\mid x_0)
=
-\frac{\epsilon}{\sigma}
}
$$

因此，可以构造 DSM Loss：

$$
\boxed{
\mathcal L_{\mathrm{DSM}}(\theta)
=
\frac12
\mathbb E_{X_0,\epsilon}
\left[
\left\|
s_\theta(X_0+\sigma\epsilon,\sigma)
+
\frac{\epsilon}{\sigma}
\right\|^2
\right]
}
$$

但是这里存在一个关键问题：

**训练监督是条件 Score，最终学习目标却是边缘 Score。为什么两者能够对应？**

### 13.3 首先推导边缘 Score 与条件 Score 的关系

从边缘分布出发：

$$
p_\sigma(x)
=
\int p_{\mathrm{data}}(x_0)p(x\mid x_0)\,dx_0
$$

对 $x$ 求梯度：

$$
\nabla_xp_\sigma(x)
=
\int
p_{\mathrm{data}}(x_0)
\nabla_xp(x\mid x_0)\,dx_0
$$

利用：

$$
\nabla p=p\nabla\log p
$$

得到：

$$
\nabla_xp_\sigma(x)
=
\int
p_{\mathrm{data}}(x_0)
p(x\mid x_0)
\nabla_x\log p(x\mid x_0)\,dx_0
$$

两边除以：

$$
p_\sigma(x)
$$

得到：

$$
\nabla_x\log p_\sigma(x)
=
\int
\frac{
p_{\mathrm{data}}(x_0)p(x\mid x_0)
}{
p_\sigma(x)
}
\nabla_x\log p(x\mid x_0)\,dx_0
$$

根据贝叶斯公式：

$$
p(x_0\mid x)
=
\frac{
p_{\mathrm{data}}(x_0)p(x\mid x_0)
}{
p_\sigma(x)
}
$$

因此：

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\mathbb E_{X_0\mid X=x}
\left[
\nabla_x\log p(x\mid X_0)
\right]
}
$$

代入条件高斯 Score：

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\mathbb E
\left[
-\frac{x-X_0}{\sigma^2}
\;\middle|\;
X=x
\right]
}
$$

整理：

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\frac{
\mathbb E[X_0\mid X=x]-x
}{
\sigma^2
}
}
$$

这也是 Tweedie 公式的一种表达形式。

它说明：

**边缘 Score 等于所有可能干净样本对应的条件 Score，在给定加噪样本后的后验加权平均。**

注意，不是简单地对整个数据集中的条件 Score 进行均匀平均。

### 13.4 为什么训练时不需要显式计算后验？

我们前面推导了：

$$
\nabla_x\log p_\sigma(x)
=
\mathbb E[
\nabla_x\log p(x\mid X_0)
\mid X=x
]
$$

但是训练时，我们并不会显式计算：

$$
p(x_0\mid x)
$$

也不会固定某个 $x$，然后从后验中采样大量 $x_0$。

实际训练只是：

$$
X_0\sim p_{\mathrm{data}}
$$

$$
\epsilon\sim\mathcal N(0,I)
$$

$$
X=X_0+\sigma\epsilon
$$

然后计算：

$$
\left\|
s_\theta(X,\sigma)+\frac{\epsilon}{\sigma}
\right\|^2
$$

这是因为：

**从 $p_{\mathrm{data}}(x_0)$ 采样，再从 $p(x\mid x_0)$ 采样，得到的样本对 $(X_0,X)$ 本身就服从正确的联合分布。**

而联合分布可以写成：

$$
p(x_0,x)
=
p_{\mathrm{data}}(x_0)p(x\mid x_0)
$$

也可以写成：

$$
p(x_0,x)
=
p_\sigma(x)p(x_0\mid x)
$$

所以：

$$
\boxed{
\mathbb E_{X_0,X}[f(X_0,X)]
=
\mathbb E_X
\left[
\mathbb E_{X_0\mid X}[f(X_0,X)]
\right]
}
$$

这就是全期望公式。

### 13.5 DSM Loss 为什么是双重期望？

定义：

$$
T(X,X_0)
=
\nabla_X\log p(X\mid X_0)
$$

那么：

$$
\mathcal L_{\mathrm{DSM}}
=
\frac12
\mathbb E_{X_0,X}
\left[
\|s_\theta(X)-T(X,X_0)\|^2
\right]
$$

根据全期望公式：

$$
\boxed{
\mathcal L_{\mathrm{DSM}}
=
\frac12
\mathbb E_{X\sim p_\sigma}
\left[
\mathbb E_{X_0\mid X}
\left[
\|s_\theta(X)-T(X,X_0)\|^2
\right]
\right]
}
$$

这里：

- 外层期望：对加噪样本 $X$ 的边缘分布取期望。
- 内层期望：固定加噪样本 $X=x$，对可能的干净样本 $X_0$ 取条件期望。

但是必须注意：

**内层期望本身是条件期望下的平方误差，不是边缘 Score。**

真正与边缘 Score 对应的是内层平方误差的最优预测值。

### 13.6 从 MSE 最优解证明 DSM 的正确性

对于固定的加噪输入：

$$
X=x
$$

定义：

$$
m(x)=\mathbb E[T(X,X_0)\mid X=x]
$$

考虑条件 MSE：

$$
\mathbb E[
\|s_\theta(x)-T\|^2
\mid X=x
]
$$

在括号中加减 $m(x)$：

$$
s_\theta(x)-T
=
[s_\theta(x)-m(x)]+[m(x)-T]
$$

展开平方并取条件期望：

$$
\begin{aligned}
&\mathbb E[
\|s_\theta(x)-T\|^2
\mid X=x
]\\
={}&
\|s_\theta(x)-m(x)\|^2\\
&+
\mathbb E[
\|T-m(x)\|^2
\mid X=x
]
\end{aligned}
$$

其中交叉项为零，因为：

$$
\mathbb E[T-m(x)\mid X=x]=0
$$

因此，条件 MSE 的最优预测为：

$$
\boxed{
s_\theta^*(x)=m(x)
}
$$

也就是：

$$
\boxed{
s_\theta^*(x)
=
\mathbb E[T\mid X=x]
}
$$

根据前面推导的边缘 Score 恒等式：

$$
\boxed{
s_\theta^*(x,\sigma)
=
\nabla_x\log p_\sigma(x)
}
$$

进一步，对 $X$ 取期望：

$$
\boxed{
\mathcal L_{\mathrm{DSM}}(\theta)
=
\frac12
\mathbb E_X
\left[
\|s_\theta(X)-\nabla\log p_\sigma(X)\|^2
\right]
+C
}
$$

其中：

$$
C
=
\frac12
\mathbb E_X
\left[
\mathbb E[
\|T-\mathbb E[T\mid X]\|^2
\mid X
]
\right]
$$

与模型参数无关。

因此：

$$
\boxed{
\arg\min_\theta\mathcal L_{\mathrm{DSM}}
=
\arg\min_\theta
\mathbb E_X
\|s_\theta(X)-\nabla\log p_\sigma(X)\|^2
}
$$

这里有一个值得强调的结论：

**DSM 与直接监督边缘 Score 的平方误差目标只相差一个与模型参数无关的常数，因此两者具有相同的参数最优解。**

这一点即使在模型容量有限时仍然成立，只要讨论的是同一模型函数类中的最优参数。

但有限容量模型不一定能够精确表示真实边缘 Score。

---

## 十四、为什么直接学习原始数据 Score 会存在困难？

论文主要讨论两个问题。

### 14.1 低维流形问题

真实数据通常具有很高的表示维度。

例如一张图像：

$$
x\in\mathbb R^{3HW}
$$

但图像内部具有大量结构约束。

例如：

- 相邻像素之间存在相关性。
- 物体形状受到几何约束。
- 光照和颜色存在统计规律。
- 真实图像不可能任意组合所有像素值。

因此，真实数据可能主要集中在高维空间中的低维流形附近。

如果数据分布严格集中在一个低维流形上，那么它相对于整个高维空间的 Lebesgue 测度，可能不存在普通的概率密度。

因此：

$$
\nabla_x\log p_{\mathrm{data}}(x)
$$

也可能无法作为整个空间上的普通 Score 定义。

通过高斯加噪：

$$
X=X_0+\sigma\epsilon
$$

得到：

$$
p_\sigma=p_{\mathrm{data}}*\mathcal N(0,\sigma^2I)
$$

对于：

$$
\sigma>0
$$

高斯卷积能够构造全空间上严格为正且平滑的密度。

因此：

$$
\boxed{
\nabla_x\log p_\sigma(x)
}
$$

可以正常定义。

这就是多尺度高斯扰动的第一个作用。

### 14.2 低概率区域的 Score 估计问题

考虑 Score 回归目标：

$$
\mathcal L(\theta)
=
\mathbb E_{x\sim p}
\|s_\theta(x)-s^*(x)\|^2
$$

写成积分：

$$
\boxed{
\mathcal L(\theta)
=
\int
p(x)\|s_\theta(x)-s^*(x)\|^2dx
}
$$

注意其中的权重：

$$
p(x)
$$

这意味着：

- 高概率区域经常出现训练样本。
- 低概率区域很少出现训练样本。
- 低概率区域的预测误差对整体 Loss 的贡献较小。

因此，模型可能在高概率区域预测准确，但在低概率区域存在明显误差。

这个问题会直接影响 Langevin 采样。

因为生成时：

$$
x_{k+1}
=
x_k+
\frac{\eta}{2}s_\theta(x_k)
+
\sqrt{\eta}z_k
$$

采样轨迹可能经过训练数据很少覆盖的位置。

如果这些位置的 Score 预测不准确，就可能导致样本移动方向错误。

### 14.3 为什么即使 Score 完全准确，采样也可能失败？

考虑两个相距很远的模态：

$$
p(x)=\pi p_1(x)+(1-\pi)p_2(x)
$$

如果两个模态之间的概率密度非常低，那么 Langevin Dynamics 很难跨越中间的概率势垒。

因此，即使：

$$
s_\theta(x)=\nabla\log p(x)
$$

完全准确，有限时间内仍然可能出现模态混合不足。

这说明：

**Score 估计误差和 Langevin 采样混合缓慢是两个不同的问题。**

前者属于模型训练问题。

后者属于采样动力学问题。

NCSN 的多尺度噪声设计同时有助于缓解两者。

---

## 十五、NCSN：为什么需要多个噪声尺度？

作者定义：

$$
\boxed{
\sigma_1>\sigma_2>\cdots>\sigma_L>0
}
$$

每个尺度对应：

$$
\boxed{
p_{\sigma_i}
=
p_{\mathrm{data}}*
\mathcal N(0,\sigma_i^2I)
}
$$

随着 $\sigma_i$ 增大：

- 原始尖峰逐渐变宽。
- 高概率峰值通常降低。
- 低概率区域得到更多概率质量。
- 不同模态之间的概率屏障可能降低。
- 分布整体更加平滑。

因此，大噪声尺度适合学习较平滑的概率结构。

小噪声尺度则更加接近原始数据分布。

NCSN 使用共享参数的网络：

$$
\boxed{
s_\theta(x,\sigma_i)
}
$$

学习：

$$
\boxed{
s_\theta(x,\sigma_i)
\approx
\nabla_x\log p_{\sigma_i}(x)
}
$$

注意：

**不同噪声尺度对应不同的目标分布，因此网络学习的并不是同一个 Score 的不同精度版本，而是一系列不同概率分布的 Score。**

---

## 十六、NCSN 的多尺度 DSM Loss

对于噪声尺度：

$$
\sigma_i
$$

定义：

$$
X=X_0+\sigma_i\epsilon
$$

条件 Score 为：

$$
-\frac{\epsilon}{\sigma_i}
$$

因此单尺度损失：

$$
\boxed{
\ell(\theta;\sigma_i)
=
\frac12
\mathbb E_{X_0,\epsilon}
\left[
\left\|
s_\theta(X_0+\sigma_i\epsilon,\sigma_i)
+
\frac{\epsilon}{\sigma_i}
\right\|^2
\right]
}
$$

多尺度损失：

$$
\boxed{
\mathcal L(\theta)
=
\frac1L
\sum_{i=1}^L
\lambda(\sigma_i)\ell(\theta;\sigma_i)
}
$$

论文采用：

$$
\boxed{
\lambda(\sigma_i)=\sigma_i^2
}
$$

原因是条件 Score：

$$
-\frac{\epsilon}{\sigma_i}
$$

的平方范数期望为：

$$
\mathbb E
\left\|
\frac{\epsilon}{\sigma_i}
\right\|^2
=
\frac D{\sigma_i^2}
$$

因此，小噪声尺度下的监督目标通常具有更大的数值量级。

使用：

$$
\lambda(\sigma_i)=\sigma_i^2
$$

能够平衡不同尺度的损失量级。

最终：

$$
\boxed{
\mathcal L(\theta)
=
\frac1{2L}
\sum_{i=1}^L
\mathbb E_{X_0,\epsilon}
\left[
\left\|
\sigma_i s_\theta(X_0+\sigma_i\epsilon,\sigma_i)
+
\epsilon
\right\|^2
\right]
}
$$

这是 NCSN 的核心训练目标。

---

## 十七、Annealed Langevin Dynamics：如何利用多个 Score 生成数据？

训练完成后，我们得到：

$$
s_\theta(x,\sigma_i)
$$

对于固定的噪声尺度 $\sigma_i$，构造：

$$
\boxed{
dY_\tau=
\frac12s_\theta(Y_\tau,\sigma_i)\,d\tau
+
dW_\tau
}
$$

如果 Score 精确，并满足适当的收敛条件，那么对应的目标平稳分布是：

$$
p_{\sigma_i}
$$

但是，如果直接在最小噪声尺度下采样：

$$
\sigma_L
$$

就会重新遇到复杂多模态分布中的混合困难。

因此，NCSN 使用退火采样。

### 17.1 从大噪声开始

首先从简单分布初始化：

$$
x^{(0)}\sim p_{\mathrm{init}}
$$

然后使用最大噪声尺度：

$$
\sigma_1
$$

执行：

$$
\boxed{
x_{k+1}
=
x_k+
\frac{\eta_1}{2}s_\theta(x_k,\sigma_1)
+
\sqrt{\eta_1}z_k
}
$$

执行若干步后，当前样本分布趋向较平滑的：

$$
p_{\sigma_1}
$$

### 17.2 逐渐降低噪声尺度

随后切换：

$$
\sigma_1\rightarrow\sigma_2
$$

继续执行：

$$
\boxed{
x_{k+1}
=
x_k+
\frac{\eta_2}{2}s_\theta(x_k,\sigma_2)
+
\sqrt{\eta_2}z_k
}
$$

然后：

$$
\sigma_2\rightarrow\sigma_3
$$

依次进行：

$$
\boxed{
\sigma_1>\sigma_2>\cdots>\sigma_L
}
$$

最终样本接近：

$$
p_{\sigma_L}
$$

当：

$$
\sigma_L
$$

足够小时：

$$
p_{\sigma_L}\approx p_{\mathrm{data}}
$$

### 17.3 采样时到底在减少什么噪声？

这里需要区分两种噪声。

**第一种：训练时的高斯扰动尺度**

$$
\sigma_i
$$

它决定当前 Score Network 对应的目标分布：

$$
p_{\sigma_i}
$$

**第二种：Langevin 采样中的随机扰动**

$$
\sqrt{\eta_i}z_k
$$

它来自布朗运动的数值离散化。

这两种噪声并不是同一个概念。

因此：

**NCSN 并不是每一步显式减去当前样本中的噪声，而是逐渐改变 Score 对应的目标分布，并通过 Langevin Dynamics 调整样本。**

### 17.4 为什么步长随噪声尺度平方变化？

论文使用：

$$
\boxed{
\eta_i
=
\varepsilon
\frac{\sigma_i^2}{\sigma_L^2}
}
$$

其中：

$$
\varepsilon
$$

是基础步长。

Langevin 更新：

$$
\Delta x
=
\frac{\eta_i}{2}s_\theta(x,\sigma_i)
+
\sqrt{\eta_i}z
$$

假设 Score 的典型范数大致满足：

$$
\|s_\theta(x,\sigma_i)\|
\propto\frac1{\sigma_i}
$$

那么漂移项的典型大小为：

$$
\frac{\eta_i}{\sigma_i}
$$

随机项的典型大小为：

$$
\sqrt{\eta_i}
$$

两者之比大致为：

$$
\frac{\sqrt{\eta_i}}{\sigma_i}
$$

如果希望不同噪声尺度下两者的相对大小大致稳定，就需要：

$$
\boxed{
\eta_i\propto\sigma_i^2
}
$$

因此，随着噪声尺度减小：

$$
\sigma_i\downarrow
$$

Langevin 步长也减小：

$$
\eta_i\downarrow
$$

这使采样过程从较大范围的随机探索逐渐过渡到更加精细的局部调整。

---

# 第四部分：NCSN 与后续扩散模型的关系

## 十八、NCSN 的 Langevin SDE 不是前向加噪 SDE

这一点非常重要。

NCSN 的高斯加噪：

$$
\boxed{
X=X_0+\sigma\epsilon
}
$$

定义的是一个条件扰动分布。

而 Langevin SDE：

$$
\boxed{
dY_\tau=
\frac12\nabla\log p_\sigma(Y_\tau)\,d\tau+dW_\tau
}
$$

是为了从固定的 $p_\sigma$ 中采样而构造的随机过程。

它们并不是同一个随机过程。

特别地，Langevin SDE 不是对上述单次高斯加噪进行时间反演得到的。

### 18.1 高斯加噪也可以用 SDE 描述

假设定义一个随时间变化的噪声标准差：

$$
\sigma(t)
$$

并满足：

$$
\sigma(0)=0
$$

考虑前向 SDE：

$$
\boxed{
dX_t=g(t)dW_t
}
$$

如果选择：

$$
\boxed{
g^2(t)=\frac{d\sigma^2(t)}{dt}
}
$$

那么：

$$
X_t=X_0+\int_0^tg(u)dW_u
$$

由于随机积分服从高斯分布：

$$
\int_0^tg(u)dW_u
\sim
\mathcal N
\left(
0,\int_0^tg^2(u)du\,I
\right)
$$

因此：

$$
\boxed{
X_t\mid X_0=x_0
\sim
\mathcal N(x_0,\sigma^2(t)I)
}
$$

这说明，前向 SDE 可以构造与高斯加噪相同的时间边缘分布。

但需要注意：

**具有相同的时间边缘分布，不意味着随机过程的完整轨迹分布相同。**

### 18.2 后来的 Score-Based SDE

在后来的 Score-Based SDE 框架中，先定义：

$$
\boxed{
dX_t=f(X_t,t)dt+g(t)dW_t
}
$$

然后通过时间反演理论得到反向 SDE：

$$
\boxed{
dX_t=
\left[
f(X_t,t)
-
g^2(t)\nabla_x\log p_t(X_t)
\right]dt
+
g(t)d\bar W_t
}
$$

这里时间从 $T$ 向 $0$ 积分。

与 NCSN 不同的是：

**反向 SDE 的目标是沿时间反转前向扩散过程，而不是让某个固定的目标分布成为采样动力学的平稳分布。**

因此，两者的数学出发点不同。

| 对比 | NCSN（2019） | Score-Based SDE（2021） |
|---|---|---|
| 训练目标 | 多尺度加噪分布的 Score | 连续时间边缘分布的 Score |
| 生成方法 | Annealed Langevin Dynamics | 反向 SDE |
| 数学机制 | 固定噪声尺度下的平稳分布采样 | 前向随机过程的时间反演 |
| 噪声变化 | 离散切换噪声尺度 | 连续时间噪声调度 |
| 是否要求目标为平稳分布 | 是，针对各固定尺度的 Langevin 动力学 | 不要求 |
| 是否直接使用反向时间 SDE | 否 | 是 |

---

# 第五部分：论文的核心结论与理论边界

## 十九、NCSN 的三项核心数学基础

### 19.1 Score Matching

通过分部积分消去未知的真实 Score：

$$
\boxed{
\mathcal J_{\mathrm{SM}}
=
\mathbb E_p
\left[
\frac12\|s_\theta\|^2
+
\nabla\cdot s_\theta
\right]
}
$$

它解决的是：

**不知道真实概率密度，如何学习概率密度梯度？**

### 19.2 Denoising Score Matching

通过已知的条件高斯 Score，学习未知的加噪边缘 Score：

$$
\boxed{
\nabla_x\log p_\sigma(x)
=
\mathbb E
\left[
-\frac{x-X_0}{\sigma^2}
\;\middle|\;
X=x
\right]
}
$$

它解决的是：

**为什么可以使用每次加噪产生的条件 Score 作为训练监督？**

### 19.3 Langevin Dynamics

通过 Score 构造具有目标平稳分布的随机动力学：

$$
\boxed{
dX_t=
\frac12\nabla\log p(X_t)\,dt+dW_t
}
$$

它解决的是：

**为什么不需要知道概率密度的具体数值，也能够从目标分布中采样？**

---

## 二十、整篇论文最重要的理论认识

我认为这篇论文真正值得掌握的，不只是最终的 DSM Loss 和采样公式，而是下面这条完整的数学逻辑。

首先，真实数据分布未知：

$$
p_{\mathrm{data}}(x)
$$

但是我们可以从中采样。

通过独立高斯扰动：

$$
X=X_0+\sigma\epsilon
$$

构造平滑分布：

$$
\boxed{
p_\sigma
=
p_{\mathrm{data}}*
\mathcal N(0,\sigma^2I)
}
$$

虽然这个边缘分布仍然难以显式计算，但条件分布：

$$
p(x\mid x_0)
$$

是已知高斯分布。

因此，我们可以解析计算：

$$
\boxed{
\nabla_x\log p(x\mid x_0)
=
-\frac{x-x_0}{\sigma^2}
}
$$

再通过 DSM 的条件期望性质：

$$
\boxed{
\mathbb E[
\nabla_x\log p(X\mid X_0)
\mid X=x
]
=
\nabla_x\log p_\sigma(x)
}
$$

训练得到：

$$
\boxed{
s_\theta(x,\sigma)
\approx
\nabla_x\log p_\sigma(x)
}
$$

然后利用 Fokker–Planck 方程构造 Langevin SDE：

$$
\boxed{
dY_\tau
=
\frac12s_\theta(Y_\tau,\sigma)\,d\tau+dW_\tau
}
$$

在理想条件下，它以：

$$
p_\sigma
$$

为目标平稳分布。

最后，通过：

$$
\sigma_1>\sigma_2>\cdots>\sigma_L
$$

逐级切换目标分布，使采样过程逐渐接近真实数据分布。

因此，整个理论链条是：

$$
\boxed{
\begin{aligned}
&\text{独立随机变量求和}\\
&\Downarrow\\
&\text{概率密度卷积}\\
&\Downarrow\\
&\text{高斯平滑数据分布}\\
&\Downarrow\\
&\text{条件分布与边缘分布}\\
&\Downarrow\\
&\text{DSM 学习边缘 Score}\\
&\Downarrow\\
&\text{利用 Score 构造 Langevin SDE}\\
&\Downarrow\\
&\text{Fokker--Planck 方程保证目标平稳分布}\\
&\Downarrow\\
&\text{Euler--Maruyama 数值采样}\\
&\Downarrow\\
&\text{多尺度退火采样生成数据}
\end{aligned}
}
$$

### 最后需要特别记住的四个区别

**第一，条件 Score 不等于边缘 Score。**

$$
\nabla_x\log p(x\mid x_0)
\neq
\nabla_x\log p_\sigma(x)
$$

但：

$$
\boxed{
\mathbb E[
\nabla_x\log p(x\mid X_0)
\mid X=x
]
=
\nabla_x\log p_\sigma(x)
}
$$

**第二，DSM 的双重期望不是对 Score 简单求平均。**

训练时最小化的是：

$$
\mathbb E_X
\mathbb E_{X_0\mid X}
\left[
\|s_\theta(X)-T(X,X_0)\|^2
\right]
$$

内层是条件平方误差，而它的最优预测才是边缘 Score。

**第三，训练时的高斯加噪和 Langevin 采样中的随机扰动不是同一回事。**

前者构造训练目标分布：

$$
p_\sigma
$$

后者构造具有目标平稳分布的随机采样过程。

**第四，NCSN 的 Langevin SDE 和后来的反向 SDE 不是同一个数学过程。**

NCSN 使用的是固定目标分布下的随机采样动力学。

后来的 Score-Based SDE 则通过前向扩散过程的时间反演构造生成动力学。

---

**对这篇论文最准确的总体理解是：**

NCSN 将概率论中的高斯卷积、统计学习中的 Score Matching，以及随机过程中的 Langevin Dynamics 连接起来。它首先通过多尺度高斯扰动构造易于学习的平滑分布，再通过 DSM 学习这些分布的 Score，最后利用以对应分布为目标平稳分布的 Langevin 动力学逐级采样。

它的重要性不仅在于提出了一个能够生成图像的模型，更在于展示了：**生成建模可以不依赖显式概率密度，而是通过学习概率密度的梯度，并结合适当的随机动力学来实现。**